# 38-Fold Cross Validation with Individual 1D Datasets
基于alex的torch版本，修改为使用38个独立的1D数据集，支持38-fold交叉验证

In [ ]:
# 第一个单元格：导入必要的库和设置
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import h5py
import scipy.io
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
import json
import os
from tqdm import tqdm
import matplotlib.pyplot as plt

# 设置随机种子确保可重现性
torch.manual_seed(42)
torch.cuda.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# 第二个单元格：加载数据集索引
# 加载数据集索引JSON文件
dataset_index_path = '/Users/jannik/KAN-Brain-Single-Voxel-Segmentaion/dataset_index_validated.json'
with open(dataset_index_path, 'r') as f:
    dataset_index = json.load(f)

print(f"找到 {len(dataset_index)} 个被试")
print("被试ID列表：")
for idx in sorted(dataset_index.keys(), key=lambda x: int(x)):
    subject = dataset_index[idx]
    print(f"  {idx}: {subject['subject_id']}")

In [ ]:
# 第三个单元格：数据加载函数
def load_subject_data(filepath):
    """
    加载单个被试的1D数据
    根据mri_dataset_create_readme.md的说明：
    - multidim_data: (351, n_voxels) -> 需要转置为 (n_voxels, 351)
    - seg_one_hot: (102, n_voxels) -> 需要转置为 (n_voxels, 102)
    """
    data = {}
    with h5py.File(filepath, 'r') as f:
        for k in f.keys():
            if not k.startswith('#'):
                v = f[k][()]
                # 根据数据维度进行转置
                if k == 'multidim_data' and v.shape[0] == 351:
                    v = v.T  # (351, n_voxels) -> (n_voxels, 351)
                elif k == 'seg_one_hot' and v.shape[0] == 102:
                    v = v.T  # (102, n_voxels) -> (n_voxels, 102)
                elif k == 'region_seg':
                    v = v.flatten()  # (1, n_voxels) -> (n_voxels,)
                data[k] = v
    return data

def load_all_subjects(dataset_index, data_dir=None):
    """
    加载所有被试的数据
    返回一个字典，key是被试ID (1-38)，value是该被试的数据
    """
    all_subjects = {}
    
    for idx in tqdm(sorted(dataset_index.keys(), key=lambda x: int(x)), desc="Loading subjects"):
        subject = dataset_index[idx]
        filepath = subject['filepath']
        
        # 如果提供了本地数据目录，替换路径
        if data_dir:
            filename = subject['filename']
            filepath = os.path.join(data_dir, filename)
        
        try:
            subject_data = load_subject_data(filepath)
            all_subjects[int(idx)] = {
                'data': subject_data.get('multidim_data'),
                'labels': subject_data.get('seg_one_hot'),
                'subject_id': subject['subject_id'],
                'prob_idx': subject['prob_idx']
            }
            print(f"  Loaded subject {idx}: {subject['subject_id']}, shape: {subject_data.get('multidim_data').shape}")
        except Exception as e:
            print(f"  Failed to load subject {idx}: {subject['subject_id']}, error: {e}")
            
    return all_subjects

In [ ]:
# 第四个单元格：加载所有数据
# 如果数据在本地，请修改这个路径
LOCAL_DATA_DIR = None  # 例如: '/path/to/local/1D/data'

# 加载所有被试数据
print("开始加载所有被试数据...")
all_subjects_data = load_all_subjects(dataset_index, LOCAL_DATA_DIR)
print(f"\n成功加载 {len(all_subjects_data)} 个被试的数据")

In [ ]:
# 第五个单元格：定义模型（与原版本相同）
# L2正则化 - 只对权重矩阵，不包括偏置（完全模拟TensorFlow的kernel_regularizer）
def kernel_l2_regularization(model, weight_decay=0.00001):
    l2_reg = 0
    for name, param in model.named_parameters():
        # 只对权重矩阵应用L2正则化，跳过偏置项
        if 'weight' in name and param.requires_grad:
            l2_reg += torch.norm(param, p=2) ** 2
    return weight_decay * l2_reg

# 定义模型（严格对应TensorFlow版本）
class RegModel(nn.Module):
    def __init__(self, input_dim=351, num_classes=102):  # 注意：输入维度改为351
        super(RegModel, self).__init__()
        # 对应TensorFlow的Dense层，使用默认初始化
        self.fc1 = nn.Linear(input_dim, 4096)
        self.fc2 = nn.Linear(4096, 4096) 
        self.fc3 = nn.Linear(4096, 4096)
        self.fc4 = nn.Linear(4096, 4096)
        self.fc5 = nn.Linear(4096, num_classes)  # visualized_layer对应的层
        self.dropout = nn.Dropout(0.5)
        
    def forward(self, x):
        # 严格按照TensorFlow模型的结构
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.dropout(F.relu(self.fc2(x)))
        x = self.dropout(F.relu(self.fc3(x)))
        x = self.dropout(F.relu(self.fc4(x)))
        x = self.fc5(x)  # 注意：这里不应用softmax，让CrossEntropyLoss处理
        return x

def create_reg_model(input_dim=351):
    model = RegModel(input_dim=input_dim).to(device)
    return model

In [ ]:
# 第六个单元格：准备训练和测试数据的函数
def prepare_fold_data(all_subjects_data, test_subject_id, val_subject_id=None, train_ratio=0.99):
    """
    准备一个fold的训练和测试数据
    
    Args:
        all_subjects_data: 所有被试的数据字典
        test_subject_id: 测试集被试ID
        val_subject_id: 验证集被试ID (如果为None，则从训练集中分割)
        train_ratio: 当val_subject_id为None时，训练集占比
    
    Returns:
        X_train, y_train, X_val, y_val, X_test, y_test, scaler
    """
    # 收集训练数据（除了测试集和验证集）
    train_data_list = []
    train_labels_list = []
    
    for subject_id, subject_data in all_subjects_data.items():
        if subject_id != test_subject_id and subject_id != val_subject_id:
            if subject_data['data'] is not None and subject_data['labels'] is not None:
                train_data_list.append(subject_data['data'])
                train_labels_list.append(subject_data['labels'])
    
    # 合并训练数据
    X_train_all = np.vstack(train_data_list)
    y_train_all = np.vstack(train_labels_list)
    
    # 如果没有指定验证集，从训练集中分割
    if val_subject_id is None:
        X_train, X_val, y_train, y_val = train_test_split(
            X_train_all, y_train_all, 
            train_size=train_ratio, 
            random_state=42
        )
    else:
        X_train = X_train_all
        y_train = y_train_all
        X_val = all_subjects_data[val_subject_id]['data']
        y_val = all_subjects_data[val_subject_id]['labels']
    
    # 获取测试数据
    X_test = all_subjects_data[test_subject_id]['data']
    y_test = all_subjects_data[test_subject_id]['labels']
    
    # 标准化
    scaler = StandardScaler()
    scaler.fit(X_train)
    X_train = scaler.transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)
    
    return X_train, y_train, X_val, y_val, X_test, y_test, scaler

In [ ]:
# 第七个单元格：训练函数（增强版，包括macro-F1监控和详细指标）
from sklearn.metrics import classification_report, precision_recall_fscore_support

def calculate_macro_f1(y_true, y_pred, num_classes=102):
    """
    Calculate macro-F1 score
    """
    return f1_score(y_true, y_pred, average='macro', labels=list(range(num_classes)), zero_division=0)

def calculate_detailed_metrics(y_true, y_pred, num_classes=102):
    """
    Calculate detailed classification metrics
    """
    # Calculate per-class precision, recall, f1-score
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, 
        labels=list(range(num_classes)),
        zero_division=0
    )
    
    # Calculate various averages
    metrics = {
        'per_class': {
            'precision': precision.tolist(),
            'recall': recall.tolist(),
            'f1': f1.tolist(),
            'support': support.tolist()
        },
        'macro_avg': {
            'precision': np.mean(precision),
            'recall': np.mean(recall),
            'f1': np.mean(f1)
        },
        'micro_avg': {},
        'weighted_avg': {}
    }
    
    # Micro average (equivalent to overall accuracy)
    micro_precision = precision_recall_fscore_support(
        y_true, y_pred, average='micro', zero_division=0
    )
    metrics['micro_avg'] = {
        'precision': micro_precision[0],
        'recall': micro_precision[1],
        'f1': micro_precision[2]
    }
    
    # Weighted average
    weighted_metrics = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )
    metrics['weighted_avg'] = {
        'precision': weighted_metrics[0],
        'recall': weighted_metrics[1],
        'f1': weighted_metrics[2]
    }
    
    # Gross accuracy (overall accuracy)
    metrics['gross_accuracy'] = np.mean(y_true == y_pred)
    
    return metrics

def print_detailed_metrics(metrics, epoch=None, dataset_name='Test'):
    """
    Print detailed classification metrics
    """
    if epoch is not None:
        print(f"\n=== Epoch {epoch} - {dataset_name} Set Detailed Metrics ===")
    else:
        print(f"\n=== {dataset_name} Set Detailed Metrics ===")
    
    print(f"\nGross Accuracy (Overall): {metrics['gross_accuracy']:.4f}")
    print(f"\nMacro Average - P: {metrics['macro_avg']['precision']:.4f}, R: {metrics['macro_avg']['recall']:.4f}, F1: {metrics['macro_avg']['f1']:.4f}")
    print(f"Micro Average - P: {metrics['micro_avg']['precision']:.4f}, R: {metrics['micro_avg']['recall']:.4f}, F1: {metrics['micro_avg']['f1']:.4f}")
    print(f"Weighted Average - P: {metrics['weighted_avg']['precision']:.4f}, R: {metrics['weighted_avg']['recall']:.4f}, F1: {metrics['weighted_avg']['f1']:.4f}")
    
    # Print classes with zero support
    zero_support_classes = [i for i, s in enumerate(metrics['per_class']['support']) if s == 0]
    if zero_support_classes:
        print(f"\nNote: The following classes have no samples in {dataset_name} set (support=0): {zero_support_classes}")
        print("Precision/Recall/F1 for these classes return 0")

def train_model_with_metrics(model, X_train, y_train, X_val, y_val, X_test, y_test,
                            batch_size=128, no_epochs=25, learning_rate=0.00001,
                            verbose=True, plot_metrics=True, print_detailed=True):
    """
    Train model and monitor macro-F1 and loss for train, validation and test sets per epoch
    """
    # Convert to PyTorch tensors
    X_train_tensor = torch.FloatTensor(X_train).to(device)
    y_train_tensor = torch.FloatTensor(y_train).to(device)
    X_val_tensor = torch.FloatTensor(X_val).to(device)
    y_val_tensor = torch.FloatTensor(y_val).to(device)
    X_test_tensor = torch.FloatTensor(X_test).to(device)
    y_test_tensor = torch.FloatTensor(y_test).to(device)

    # Create data loader
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # Compile model
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    # Training history
    history = {
        'train_loss': [], 'train_acc': [], 'train_f1': [],
        'val_loss': [], 'val_acc': [], 'val_f1': [],
        'test_loss': [], 'test_acc': [], 'test_f1': [],
        'test_detailed_metrics': []  # Detailed test metrics per epoch
    }

    # Training loop
    for epoch in range(no_epochs):
        # ========== Training Phase ==========
        model.train()
        epoch_train_loss = 0
        all_train_preds = []
        all_train_labels = []

        for batch_idx, (data, target) in enumerate(train_loader):
            optimizer.zero_grad()

            output = model(data)
            target_indices = torch.argmax(target, dim=1)

            # Calculate loss
            base_loss = criterion(output, target_indices)
            l2_reg = kernel_l2_regularization(model, weight_decay=0.00001)
            total_loss = base_loss + l2_reg

            total_loss.backward()
            optimizer.step()

            epoch_train_loss += total_loss.item()

            # Collect predictions
            _, predicted = torch.max(output.data, 1)
            all_train_preds.extend(predicted.cpu().numpy())
            all_train_labels.extend(target_indices.cpu().numpy())

        # Calculate training metrics
        avg_train_loss = epoch_train_loss / len(train_loader)
        train_acc = np.mean(np.array(all_train_preds) == np.array(all_train_labels))
        train_f1 = calculate_macro_f1(all_train_labels, all_train_preds)

        # ========== Validation Phase ==========
        model.eval()
        with torch.no_grad():
            # Validation set
            val_output = model(X_val_tensor)
            val_target_indices = torch.argmax(y_val_tensor, dim=1)
            val_base_loss = criterion(val_output, val_target_indices)
            val_l2_reg = kernel_l2_regularization(model, weight_decay=0.00001)
            val_total_loss = val_base_loss + val_l2_reg

            _, val_predicted = torch.max(val_output.data, 1)
            val_acc = (val_predicted == val_target_indices).float().mean().item()
            val_f1 = calculate_macro_f1(
                val_target_indices.cpu().numpy(),
                val_predicted.cpu().numpy()
            )

            # Test set
            test_output = model(X_test_tensor)
            test_target_indices = torch.argmax(y_test_tensor, dim=1)
            test_base_loss = criterion(test_output, test_target_indices)
            test_l2_reg = kernel_l2_regularization(model, weight_decay=0.00001)
            test_total_loss = test_base_loss + test_l2_reg

            _, test_predicted = torch.max(test_output.data, 1)
            test_labels_np = test_target_indices.cpu().numpy()
            test_preds_np = test_predicted.cpu().numpy()
            
            # Calculate detailed test metrics
            test_detailed = calculate_detailed_metrics(test_labels_np, test_preds_np)
            test_acc = test_detailed['gross_accuracy']  # Use gross accuracy
            test_f1 = test_detailed['macro_avg']['f1']

        # Record history
        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_acc)
        history['train_f1'].append(train_f1)
        history['val_loss'].append(val_total_loss.item())
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        history['test_loss'].append(test_total_loss.item())
        history['test_acc'].append(test_acc)
        history['test_f1'].append(test_f1)
        history['test_detailed_metrics'].append(test_detailed)

        # Print progress
        if verbose and (epoch % 1 == 0 or epoch == no_epochs - 1):
            print(f"\nEpoch {epoch+1}/{no_epochs}:")
            print(f"  Train - Loss: {avg_train_loss:.4f}, Acc: {train_acc:.4f}, Macro-F1: {train_f1:.4f}")
            print(f"  Val   - Loss: {val_total_loss.item():.4f}, Acc: {val_acc:.4f}, Macro-F1: {val_f1:.4f}")
            print(f"  Test  - Loss: {test_total_loss.item():.4f}, Gross Acc: {test_acc:.4f}, Macro-F1: {test_f1:.4f}")
            
            # Print detailed metrics at specific epochs
            if print_detailed and (epoch == 0 or epoch == no_epochs - 1 or epoch % 5 == 0):
                print_detailed_metrics(test_detailed, epoch+1, 'Test')

    # Plot training curves
    if plot_metrics:
        plot_training_history(history, no_epochs)

    return history

def plot_training_history(history, no_epochs):
    """
    Plot training history curves
    """
    epochs = range(1, no_epochs + 1)

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Training History', fontsize=16)

    # Loss curves
    axes[0, 0].plot(epochs, history['train_loss'], 'b-', label='Train Loss')
    axes[0, 0].plot(epochs, history['val_loss'], 'r-', label='Val Loss')
    axes[0, 0].plot(epochs, history['test_loss'], 'g-', label='Test Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Loss over Epochs')
    axes[0, 0].legend()
    axes[0, 0].grid(True)

    # Accuracy curves
    axes[0, 1].plot(epochs, history['train_acc'], 'b-', label='Train Acc')
    axes[0, 1].plot(epochs, history['val_acc'], 'r-', label='Val Acc')
    axes[0, 1].plot(epochs, history['test_acc'], 'g-', label='Test Gross Acc')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].set_title('Accuracy over Epochs')
    axes[0, 1].legend()
    axes[0, 1].grid(True)

    # Macro-F1 curves
    axes[1, 0].plot(epochs, history['train_f1'], 'b-', label='Train Macro-F1')
    axes[1, 0].plot(epochs, history['val_f1'], 'r-', label='Val Macro-F1')
    axes[1, 0].plot(epochs, history['test_f1'], 'g-', label='Test Macro-F1')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Macro-F1')
    axes[1, 0].set_title('Macro-F1 Score over Epochs')
    axes[1, 0].legend()
    axes[1, 0].grid(True)

    # F1 vs Accuracy comparison
    axes[1, 1].plot(epochs, history['train_f1'], 'b-', label='Train F1', alpha=0.7)
    axes[1, 1].plot(epochs, history['train_acc'], 'b--', label='Train Acc', alpha=0.7)
    axes[1, 1].plot(epochs, history['test_f1'], 'g-', label='Test F1', alpha=0.7)
    axes[1, 1].plot(epochs, history['test_acc'], 'g--', label='Test Acc', alpha=0.7)
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Score')
    axes[1, 1].set_title('F1 vs Accuracy Comparison')
    axes[1, 1].legend()
    axes[1, 1].grid(True)

    plt.tight_layout()
    plt.show()

# Keep original training function for compatibility
def train_model(model, X_train, y_train, X_val, y_val,
                batch_size=128, no_epochs=25, learning_rate=0.00001):
    """
    Original training function (kept for compatibility)
    """
    # Create dummy test set (using validation set)
    history_full = train_model_with_metrics(
        model, X_train, y_train, X_val, y_val, X_val, y_val,
        batch_size, no_epochs, learning_rate,
        verbose=False, plot_metrics=False, print_detailed=False
    )

    # Return compatible history
    history = {
        'loss': history_full['train_loss'],
        'accuracy': history_full['train_acc'],
        'val_loss': history_full['val_loss'],
        'val_accuracy': history_full['val_acc']
    }
    return history

In [ ]:
# 第八个单元格：评估函数
def evaluate_model(model, X_test, y_test):
    """
    评估模型在测试集上的性能
    """
    X_test_tensor = torch.FloatTensor(X_test).to(device)
    y_test_tensor = torch.FloatTensor(y_test).to(device)
    
    model.eval()
    with torch.no_grad():
        test_output = model(X_test_tensor)
        test_target_indices = torch.argmax(y_test_tensor, dim=1)
        
        # 计算准确率
        _, test_predicted = torch.max(test_output.data, 1)
        test_correct = (test_predicted == test_target_indices).sum().item()
        test_accuracy = test_correct / test_target_indices.size(0)
        
        # 计算损失
        criterion = nn.CrossEntropyLoss()
        test_loss = criterion(test_output, test_target_indices)
    
    return test_accuracy, test_loss.item()

In [ ]:
# 第九个单元格：单个实验 - 使用被试38作为固定测试集（增强版）
print("=" * 50)
print("单个实验：被试38作为测试集（监控macro-F1和详细指标）")
print("=" * 50)

# 准备数据
X_train, y_train, X_val, y_val, X_test, y_test, scaler = prepare_fold_data(
    all_subjects_data,
    test_subject_id=38,  # 被试38作为测试集
    val_subject_id=None,  # 从训练集中分割验证集
    train_ratio=0.99
)

print(f"训练集大小: {X_train.shape}")
print(f"验证集大小: {X_val.shape}")
print(f"测试集大小: {X_test.shape}")

# 创建模型
model = create_reg_model(input_dim=351)

# 使用增强版训练函数，监控所有指标
print("\n开始训练（监控训练集、验证集和测试集）...")
print("=" * 50)
history = train_model_with_metrics(
    model, X_train, y_train, X_val, y_val, X_test, y_test,
    batch_size=128, no_epochs=25, learning_rate=0.00001,
    verbose=True, plot_metrics=True, print_detailed=True
)

# 打印最终结果和详细的分类报告
print("\n" + "=" * 50)
print("训练完成！最终结果：")
print("=" * 50)

# 获取最终的详细指标
final_metrics = history['test_detailed_metrics'][-1]

print(f"\n测试集最终性能：")
print(f"  Gross Accuracy (Overall): {final_metrics['gross_accuracy']:.4f}")
print(f"  Macro-F1: {final_metrics['macro_avg']['f1']:.4f}")
print(f"  Loss: {history['test_loss'][-1]:.4f}")

print(f"\n各种平均指标：")
print(f"  Macro Average    - Precision: {final_metrics['macro_avg']['precision']:.4f}, Recall: {final_metrics['macro_avg']['recall']:.4f}, F1: {final_metrics['macro_avg']['f1']:.4f}")
print(f"  Micro Average    - Precision: {final_metrics['micro_avg']['precision']:.4f}, Recall: {final_metrics['micro_avg']['recall']:.4f}, F1: {final_metrics['micro_avg']['f1']:.4f}")
print(f"  Weighted Average - Precision: {final_metrics['weighted_avg']['precision']:.4f}, Recall: {final_metrics['weighted_avg']['recall']:.4f}, F1: {final_metrics['weighted_avg']['f1']:.4f}")

# 生成完整的分类报告
from sklearn.metrics import classification_report
model.eval()
with torch.no_grad():
    X_test_tensor = torch.FloatTensor(X_test).to(device)
    y_test_tensor = torch.FloatTensor(y_test).to(device)
    test_output = model(X_test_tensor)
    _, test_predicted = torch.max(test_output.data, 1)
    test_labels_np = torch.argmax(y_test_tensor, dim=1).cpu().numpy()
    test_preds_np = test_predicted.cpu().numpy()

print("\n完整分类报告：")
print(classification_report(test_labels_np, test_preds_np, 
                          labels=list(range(102)), 
                          zero_division=0))

# 保存模型和训练历史
export_path = '/Users/jannik/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/erosion/'
model_save_path = os.path.join(export_path, 'model_subject38_test.pth')
torch.save(model.state_dict(), model_save_path)
print(f"\n模型已保存到: {model_save_path}")

# 保存训练历史（包含详细指标）
history_save_path = os.path.join(export_path, 'training_history_subject38_detailed.json')
# 将numpy arrays转换为lists以便JSON序列化
history_to_save = {k: v for k, v in history.items() if k != 'test_detailed_metrics'}
history_to_save['test_detailed_metrics_summary'] = {
    'final_gross_accuracy': final_metrics['gross_accuracy'],
    'final_macro_f1': final_metrics['macro_avg']['f1'],
    'final_macro_precision': final_metrics['macro_avg']['precision'],
    'final_macro_recall': final_metrics['macro_avg']['recall'],
    'final_micro_f1': final_metrics['micro_avg']['f1'],
    'final_weighted_f1': final_metrics['weighted_avg']['f1']
}
with open(history_save_path, 'w') as f:
    json.dump(history_to_save, f, indent=2)
print(f"训练历史已保存到: {history_save_path}")

In [ ]:
# 第十个单元格：38-fold交叉验证
def run_38fold_cross_validation(all_subjects_data, save_results=True):
    """
    运行38-fold交叉验证
    每个被试轮流作为测试集
    """
    results = []
    
    print("=" * 50)
    print("开始38-Fold交叉验证")
    print("=" * 50)
    
    for test_subject_id in sorted(all_subjects_data.keys()):
        print(f"\nFold {test_subject_id}/38: 测试被试 = {test_subject_id}")
        
        try:
            # 准备数据
            X_train, y_train, X_val, y_val, X_test, y_test, scaler = prepare_fold_data(
                all_subjects_data, 
                test_subject_id=test_subject_id,
                val_subject_id=None,  # 从训练集中分割验证集
                train_ratio=0.99
            )
            
            print(f"  训练集: {X_train.shape}, 验证集: {X_val.shape}, 测试集: {X_test.shape}")
            
            # 创建新模型
            model = create_reg_model(input_dim=351)
            
            # 训练模型
            history = train_model(
                model, X_train, y_train, X_val, y_val,
                batch_size=128, no_epochs=25, learning_rate=0.00001
            )
            
            # 评估模型
            test_acc, test_loss = evaluate_model(model, X_test, y_test)
            
            # 保存结果
            fold_result = {
                'fold': test_subject_id,
                'subject_id': all_subjects_data[test_subject_id]['subject_id'],
                'test_accuracy': test_acc,
                'test_loss': test_loss,
                'val_accuracy': history['val_accuracy'][-1],
                'val_loss': history['val_loss'][-1]
            }
            results.append(fold_result)
            
            print(f"  Fold {test_subject_id} 完成: Test Acc = {test_acc:.4f}")
            
            # 保存模型
            if save_results:
                model_save_path = os.path.join(export_path, f'model_fold{test_subject_id}.pth')
                torch.save(model.state_dict(), model_save_path)
                
        except Exception as e:
            print(f"  Fold {test_subject_id} 失败: {e}")
            results.append({
                'fold': test_subject_id,
                'subject_id': all_subjects_data[test_subject_id]['subject_id'],
                'error': str(e)
            })
    
    # 计算平均性能
    valid_results = [r for r in results if 'test_accuracy' in r]
    if valid_results:
        avg_accuracy = np.mean([r['test_accuracy'] for r in valid_results])
        std_accuracy = np.std([r['test_accuracy'] for r in valid_results])
        print(f"\n38-Fold 交叉验证结果:")
        print(f"平均准确率: {avg_accuracy:.4f} ± {std_accuracy:.4f}")
        print(f"成功折数: {len(valid_results)}/{len(results)}")
    
    # 保存结果
    if save_results:
        results_path = os.path.join(export_path, 'cross_validation_results.json')
        with open(results_path, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"\n结果已保存到: {results_path}")
    
    return results

In [ ]:
# 第十一个单元格：运行38-fold交叉验证
# 注意：这会训练38个模型，可能需要较长时间
# 如果只想测试，可以先运行单个实验（第九个单元格）

# 取消下面的注释来运行完整的38-fold交叉验证
# cv_results = run_38fold_cross_validation(all_subjects_data, save_results=True)

In [ ]:
# 第十二个单元格：可视化结果（如果运行了交叉验证）
import matplotlib.pyplot as plt

def visualize_cv_results(cv_results):
    """
    Visualize cross-validation results
    """
    valid_results = [r for r in cv_results if 'test_accuracy' in r]
    
    if not valid_results:
        print("No valid results to visualize")
        return
    
    folds = [r['fold'] for r in valid_results]
    accuracies = [r['test_accuracy'] for r in valid_results]
    
    plt.figure(figsize=(15, 6))
    
    # Bar chart
    plt.subplot(1, 2, 1)
    plt.bar(folds, accuracies)
    plt.xlabel('Subject ID')
    plt.ylabel('Test Accuracy')
    plt.title('38-Fold Cross Validation: Test Accuracy per Fold')
    plt.xticks(rotation=45)
    mean_acc = np.mean(accuracies)
    plt.axhline(y=mean_acc, color='r', linestyle='--', label=f'Mean: {mean_acc:.4f}')
    plt.legend()
    
    # Box plot
    plt.subplot(1, 2, 2)
    plt.boxplot(accuracies)
    plt.ylabel('Test Accuracy')
    plt.title('Accuracy Distribution')
    plt.xticks([])
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"\nStatistics:")
    print(f"Highest Accuracy: {np.max(accuracies):.4f} (Subject {folds[np.argmax(accuracies)]})")
    print(f"Lowest Accuracy: {np.min(accuracies):.4f} (Subject {folds[np.argmin(accuracies)]})")
    print(f"Mean Accuracy: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")

# Uncomment to visualize results if cross-validation was run
# visualize_cv_results(cv_results)